📋 [팀 공통] BaseLLMNode 기반 노드 구현 가이드

작성자: 박제혁

목적: 복잡한 모델 로딩 과정을 제거하고, BaseLLMNode를 상속받아 표준화된 방식으로 노드를 구현하는 방법을 공유합니다.

🚀 무엇이 바뀌었나요?
- 복잡한 코드 삭제: 토크나이저 로드, GPU 이동, apply_chat_template, decode 등 반복되는 코드를 BaseLLMNode 내부로 숨겼습니다.
- 설정 관리 자동화: config.yaml에 정의된 temperature, top_p 등의 설정이 코드를 수정하지 않아도 자동으로 적용됩니다.
- 메인/서브 모델 선택: main_solver (Qwen3) 또는 sub_solver (Qwen2.5)를 이름만으로 쉽게 불러올 수 있습니다.

🛠️ 사용 방법 (3 Step)
- 상속: 내 노드 클래스에 BaseLLMNode를 상속받습니다.
- 초기화: super().__init__(config, model_name="...")를 호출하여 모델을 로드합니다.
- 생성: self.generate(template, **kwargs) 함수로 답변을 생성합니다.

In [ ]:
# [참고] 구현 예시 구조
# class MyNode(BaseLLMNode):
#     def __init__(self, config):
#         # 1. 부모 클래스 초기화 (모델 로딩 자동화)
#         super().__init__(config, model_name="main_solver")
        
#         # 2. 프롬프트 템플릿 로드
#         self.template = config.prompt.my_node

#     def __call__(self, state):
#         # 3. 핵심 로직 (프롬프트 변수만 채워주면 됨)
#         answer = self.generate(self.template, question=state["question"])
#         return {"result": answer}

In [ ]:
# [Cell 1] 환경 설정 및 임포트
import sys
import os

# 1. 프로젝트 루트 경로 추가 (notebooks 폴더 기준 상위 폴더)
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. 필요한 모듈 임포트
from src.utils.config_loader import load_config
from src.agent.nodes.base import BaseLLMNode

# 3. 설정 로드 
cfg = load_config()

print("✅ 설정 및 라이브러리 로드 완료!")
print(f"🔹 사용 모델: {cfg.model.main_solver.path}")

✅ 설정 및 라이브러리 로드 완료!
🔹 사용 모델: unsloth/Qwen3-32B-bnb-4bit


In [ ]:
# [Cell 2] 나만의 노드 만들기 (팀원들이 해야 할 작업)

class MyTestNode(BaseLLMNode):
    def __init__(self, config):
        # 1. 부모 클래스 초기화 (모델 로딩)
        # model_name="main_solver" 또는 "sub_solver" 선택 가능
        super().__init__(config, model_name="main_solver")
        
        # 2. 테스트용 프롬프트 템플릿 정의
        # (실제로는 config.prompt.xxx.template 에서 가져오면 됩니다)
        self.template = """
        당신은 친절한 AI 조교입니다.
        사용자의 질문에 대해 핵심만 요약해서 답변하세요.
        
        질문: {question}
        요약 답변:
        """

    def __call__(self, state: dict) -> dict:
        """LangGraph가 실행할 함수"""
        print(f"▶️ 노드 실행 중... 질문: {state['question']}")
        
        # 3. generate 함수 호출 (핵심!)
        # 템플릿에 있는 {question} 변수만 채워주면 끝
        answer = self.generate(self.template, question=state["question"])
        
        return {"answer": answer}

print("✅ MyTestNode 클래스 정의 완료! (모델은 아직 로드 안 됨)")

✅ MyTestNode 클래스 정의 완료! (모델은 아직 로드 안 됨)


In [3]:
# [Cell 3] 노드 실행 테스트
# 실제 모델이 메모리에 올라가고 추론이 진행됩니다.

# 1. 노드 생성 (이 시점에 모델이 로딩됨 - 시간 소요)
print("⏳ 모델 로딩 시작... (잠시만 기다려주세요)")
node = MyTestNode(cfg)
print("✅ 모델 로딩 완료!")

# 2. 가짜 상태(State) 데이터 생성
dummy_state = {
    "question": "2024년 수능 영어 영역의 난이도가 어땠는지 알려줘."
}

# 3. 노드 실행 (추론)
result = node(dummy_state)

# 4. 결과 확인
print("\n" + "="*30)
print(f"🤖 결과:\n{result['answer']}")
print("="*30)

⏳ 모델 로딩 시작... (잠시만 기다려주세요)
🔧 [MyTestNode] 초기화 중... (사용 모델: main_solver)
🔄 [Loader] 모델 로딩 시작: Qwen3-32B-4bit (unsloth/Qwen3-32B-bnb-4bit)
   ↳ ⚡ 양자화 설정 적용 중...


/data/ephemeral/home/REPO_Jehyeok/.venv/lib/python3.11/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ [Loader] 로딩 완료!
✅ 모델 로딩 완료!
▶️ 노드 실행 중... 질문: 2024년 수능 영어 영역의 난이도가 어땠는지 알려줘.

🤖 결과:
<think>
Okay, I need to answer the user's question about the difficulty of the 2024 Korean College Scholastic Ability Test (CSAT) English section. First, I should recall any available information about the 2024 CSAT English exam. Since I don't have real-time data, I might need to refer to past trends or any official statements from the year before.

Wait, the user is asking specifically about 2024, but I don't have access to current data beyond my training cutoff in October 2023. So I can't provide actual details on the 2024 exam. I should inform the user that I can't provide real-time updates but can offer insights based on previous years or general trends.

I should mention that the English section typically includes reading comprehension, grammar, vocabulary, and listening sections. Difficulty can vary each year, but generally, the CSAT aims for a balanced level. Maybe suggest checking official sou